# Runs on A100 only
# After Cell 2 fails restart session and rerun from cell 3

## 1. Confirm the GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> A100 GPU."
name = torch.cuda.get_device_name(0)
print("GPU:", name)
if "A100" not in name:
    print("\nWarning: not an A100. The settings below assume 40GB and bfloat16.")
    print("On a T4 (no bf16, 16GB) you must switch to --dtype float16 and an AWQ build")
    print("such as TheBloke/meditron-7B-AWQ, and expect roughly 5-10x slower generation.")

name, memory.total [MiB], compute_cap
NVIDIA A100-SXM4-40GB, 40960 MiB, 8.0
GPU: NVIDIA A100-SXM4-40GB


## 2. Install vLLM

Takes 3-5 minutes. If Colab prompts you to restart the session afterwards, restart and
then re-run from cell 3 onwards — do not re-run this install.

In [2]:
!pip install -q -U vllm pyngrok
import vllm; print("vLLM", vllm.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.5/314.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5

ImportError: libcudart.so.13: cannot open shared object file: No such file or directory

## 3. Settings

`API_KEY` is a password of your choosing — the tunnel is a public URL, so without it
anyone who guesses the address gets free use of your GPU. Put the same value in the
local `.env`.

In [1]:
import vllm, torch
print("vLLM", vllm.__version__, "| torch CUDA", torch.version.cuda)

vLLM 0.28.0 | torch CUDA 13.0


In [2]:
from getpass import getpass

MODEL_ID   = "google/medgemma-4b-it"
MAX_LEN    = 8192
PORT       = 8000
API_KEY    = "pick-a-long-random-string"

HF_TOKEN    = "hf_YOUR_HUGGINGFACE_TOKEN_HERE"
NGROK_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"

import os
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

from huggingface_hub import login, model_info
login(token=HF_TOKEN, add_to_git_credential=False)
try:
    model_info(MODEL_ID)
    print(f"Access to {MODEL_ID} confirmed.")
except Exception as exc:
    print(f"Cannot access {MODEL_ID}: {exc}")
    print("Open the model page on huggingface.co and accept the licence conditions first.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Access to google/medgemma-4b-it confirmed.


## 4. Start vLLM

What each flag buys you on an A100-40GB:

| flag | why |
|---|---|
| `--dtype bfloat16` | A100 has native bf16. Same speed as fp16, no loss-scaling overflow risk. 7B weights ≈ 13.5 GB, so no quantization is needed — quantizing here would cost accuracy for memory you already have. |
| `--gpu-memory-utilization 0.92` | Leaves ~25 GB for the KV cache after weights. At 2048 tokens that is thousands of cached sequences. |
| `--enable-prefix-caching` | The real win for this app. Every turn re-sends the same system block, patient record and one-shot demo. Prefix caching skips recomputing that prefill, so follow-up questions start generating almost immediately. |
| `--max-num-seqs 64` | Continuous batching headroom. Irrelevant for one user, free to leave on. |
| CUDA graphs | On by default. Do **not** pass `--enforce-eager`; it costs roughly 20-30% decode throughput. |

First run downloads ~13 GB, so give it a few minutes.

In [3]:
import torch, torchvision
print(torch.__version__, torch.version.cuda)
print(torchvision.__version__)

2.13.0+cu130 13.0
0.28.0+cu130


In [4]:
!pip uninstall -y -q torchaudio

In [5]:
import subprocess, time, requests, os, sys

LOG = "/content/vllm.log"
cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--served-model-name", MODEL_ID,
    "--dtype", "bfloat16", #bloat16 if A100
    "--max-model-len", str(MAX_LEN),
    "--gpu-memory-utilization", "0.90",
    "--max-num-seqs", "64",
    "--no-enable-log-requests",
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--api-key", API_KEY,
]

logfile = open(LOG, "w")
server = subprocess.Popen(cmd, stdout=logfile, stderr=subprocess.STDOUT, env=os.environ.copy())
print("Loading. Watch the log below; first run downloads the weights.\n")

start = time.time()
while True:
    if server.poll() is not None:
        print("Server exited. Last 40 log lines:\n")
        print("".join(open(LOG).readlines()[-40:]))
        raise SystemExit("vLLM failed to start.")
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).status_code == 200:
            print(f"\nReady in {time.time()-start:.0f}s.")
            break
    except requests.RequestException:
        pass
    if int(time.time() - start) % 20 == 0:
        tail = open(LOG).readlines()[-1:]
        if tail:
            print("  ", tail[0].strip()[:140])
    time.sleep(5)

Loading. Watch the log below; first run downloads the weights.

   (APIServer pid=5420) INFO 08-30 10:00:39 [api_utils.py:272] non-default args: {'host': '0.0.0.0', 'api_key': ['pick-a-long-random-string'], 
   (APIServer pid=5420) WARNING 08-30 10:00:57 [cuda.py:327] Forcing --disable_chunked_mm_input for models with multimodal-bidirectional attent
   (APIServer pid=5420) WARNING 08-30 10:00:57 [cuda.py:327] Forcing --disable_chunked_mm_input for models with multimodal-bidirectional attent
   (EngineCore pid=5863) INFO 08-30 10:01:42 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=
   (EngineCore pid=5863) INFO 08-30 10:01:45 [cuda.py:486] Using TRITON_ATTN attention backend out of potential backends: ['TRITON_ATTN', 'FLEX
   (EngineCore pid=5863) INFO 08-30 10:02:15 [topk_topp_sampler.py:62] Using FlashInfer for top-p & top-k sampling.
   (EngineCore pid=5863) INFO 08-30 10:02:33 [encoder_runner.py:120] Encoder cache will be initializ

## 5. Open the tunnel

Copy the printed block straight into `meditron-local/.env`.

The free ngrok tier gives you one tunnel and a URL that changes every time you restart —
so expect to re-paste this after each Colab session. A reserved domain on a paid plan
removes that step.

In [6]:
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()
tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url.replace("http://", "https://")

print("Paste this into meditron-local/.env:\n")
print(f"COLAB_API_BASE={public_url}")
print(f"COLAB_API_KEY={API_KEY}")
print(f"MODEL_ID={MODEL_ID}")
print(f"MAX_MODEL_LEN={MAX_LEN}")

Paste this into meditron-local/.env:

COLAB_API_BASE=https://chalcographic-claudette-solemn.ngrok-free.dev
COLAB_API_KEY=pick-a-long-random-string
MODEL_ID=google/medgemma-4b-it
MAX_MODEL_LEN=8192


## 6. Test it the way the app will call it

Same ChatML shape and one-shot priming the local `memory.py` uses, so if this reads
sensibly the app will too.

In [7]:
import requests, json

prompt = """<|im_start|>system
You are a careful medical information assistant. Answer in plain language.

PATIENT RECORD
Age: 31
Existing conditions: mild asthma
Allergies: penicillin<|im_end|>
<|im_start|>question
I've had a dry cough for four days and a mild sore throat. No fever.<|im_end|>
<|im_start|>answer
A dry cough with a mild sore throat and no fever most often follows a viral upper respiratory infection, and it usually settles over one to two weeks.<|im_end|>
<|im_start|>question
My asthma inhaler helps a little but the cough is worse at night. Should I be worried?<|im_end|>
<|im_start|>answer
"""

r = requests.post(
    f"http://127.0.0.1:{PORT}/v1/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={"model": MODEL_ID, "prompt": prompt, "max_tokens": 300,
          "temperature": 0.3, "top_p": 0.9, "repetition_penalty": 1.1,
          "stop": ["<|im_end|>", "<|im_start|>"]},
    timeout=180,
)
print(r.json()["choices"][0]["text"].strip())
print("\n---\nusage:", r.json()["usage"])

It's good that your inhaler provides some relief. A nighttime cough can be common during a cold or flu because lying down can make mucus drain into your throat more easily. If you feel short of breath or have wheezing, contact your doctor right away. Otherwise, try using your inhaler before bed as directed, staying hydrated, and resting. If symptoms persist after two weeks, see your doctor.

---
usage: {'prompt_tokens': 170, 'total_tokens': 259, 'completion_tokens': 89, 'prompt_tokens_details': None, 'completion_tokens_details': None}


## 7. Keep the session awake

Colab disconnects idle notebooks. Run this cell and leave the tab open while you use the
UI. Interrupt it when you're done.

Honest note on cost: an A100 burns Colab compute units continuously while this runs,
whether or not you are asking questions. A 7B model at bf16 also fits comfortably on an
L4 or a T4-with-AWQ if you want the same app for a fraction of the units — the A100 buys
you latency, not capability, for a single user.

In [ ]:
import time, requests
from datetime import datetime

while True:
    try:
        ok = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=5).status_code == 200
    except requests.RequestException:
        ok = False
    print(f"{datetime.now():%H:%M:%S}  {'serving' if ok else 'DOWN — re-run cell 4'}   ", end="\r")
    time.sleep(60)

## 8. Shut down

In [ ]:
from pyngrok import ngrok
ngrok.kill()
try:
    server.terminate(); server.wait(timeout=30)
except Exception as exc:
    print(exc)
print("Tunnel closed, server stopped.")